# DressMe — lancer et déployer l'interface
Dépose `dressme_app.zip` à la racine de ton Drive. Secrets Colab (🔑, accès au notebook activé) : `GEMINI_API_KEY` et `LLM_API_KEY` (facultatifs), `HF_TOKEN` (pour la dernière cellule). Runtime CPU suffisant.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
ZIP = "/content/drive/MyDrive/dressme_app.zip"          # dépose le zip ici
DATA = "/content/drive/MyDrive/dressme_data"            # photos, truth.json, perception_clip.json, survey2/3
APP = "/content/dressme_app"
subprocess.run(["unzip", "-oq", ZIP, "-d", "/content"], check=True)
for k in ("GEMINI_API_KEY", "LLM_API_KEY"):             # facultatives : sans elles, règles seules
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print("secret absent :", k)
print(sorted(os.listdir(APP)))

In [ ]:
!pip -q install -r /content/dressme_app/requirements.txt huggingface_hub

In [ ]:
import json
truth = json.load(open(f"{DATA}/truth.json", encoding="utf-8"))
perc = json.load(open(f"{DATA}/perception_clip.json", encoding="utf-8"))
ok = [f for f in truth if f in perc and os.path.exists(f"{DATA}/wardrobe/{f}")]
print(len(truth), "pièces dans truth.json |", len(perc), "décrites |", len(ok), "utilisables")
print("questionnaires :", [d for d in os.listdir(DATA) if d.startswith("survey")])
assert ok, "Aucune pièce décrite : relance la perception CLIP sur toute la garde-robe (notebook v2)."

In [ ]:
!python /content/dressme_app/prepare_app_data.py {DATA} /content/dressme_app/app_data

## Lancer l'interface
Un lien public temporaire s'affiche. Laisse la cellule tourner : si tu l'interromps, le lien s'arrête.

In [ ]:
import sys, os
os.chdir(APP); sys.path.insert(0, APP)
os.environ["APP_DATA"] = APP + "/app_data"
import app
app.demo.queue(max_size=16).launch(share=True, debug=True)

## (Optionnel) Déploiement durable sur Hugging Face Spaces
Le Space est créé **privé** : vérifie la licence des photos H&M avant de le rendre public.

In [ ]:
from huggingface_hub import HfApi
HF_USER = "TON_IDENTIFIANT_HF"                          # à remplacer
api = HfApi(token=userdata.get("HF_TOKEN"))             # jeton avec droit d'écriture
repo = f"{HF_USER}/dressme"
api.create_repo(repo, repo_type="space", space_sdk="gradio", private=True, exist_ok=True)
api.upload_folder(folder_path=APP, repo_id=repo, repo_type="space",
                  ignore_patterns=["__pycache__", "*.pyc", "app_data/llm_cache/*", "app_data_demo/*", ".env"])
for k in ("GEMINI_API_KEY", "LLM_API_KEY"):
    if os.environ.get(k):
        api.add_space_secret(repo, k, os.environ[k])
print("https://huggingface.co/spaces/" + repo)